
# Chapter 1: Introduction

Source span: printed pages 1-26; PDF pages 12-37. The source was inspected for orientation only: Euler characteristic, polyhedra, homeomorphism, surfaces, classification, orientability, and invariants. The prose, examples, diagrams, computations, and artifacts here are original course material.

## Chapter Goal

The opening chapter teaches a change of viewpoint. Geometry first sees a solid, a drawing, or a familiar surface; topology asks which features survive continuous deformation. A cube and a tetrahedron look different as metric objects, but their polyhedral surfaces share the same Euler characteristic and both behave like spheres. A torus has a loop that does not separate it, so its Euler characteristic and global behavior are different. A Mobius strip warns us that a surface can look locally like a plane while reversing orientation globally.

This notebook makes that viewpoint executable. We build small combinatorial models of polyhedra and surfaces, compute `V - E + F`, track how adding handles or crosscaps changes the invariant, draw a classification decision map, and test simple topological invariants such as connectedness after puncturing or cutting. The point is not to prove the full classification theorem; it is to give the reader enough visual and computational scaffolding that later definitions feel necessary rather than ceremonial.



## Visualization Storyboard And Library Routing

- **Euler ledger.** A Matplotlib table and graph compare sphere-like polyhedra, torus-like cell structures, and disconnected examples. The inspection target is which hypotheses are needed before `V - E + F = 2` is expected.
- **Tree thickening and dual intuition.** A NetworkX diagram shows the proof-state behind Euler's theorem: a spanning tree in the primal graph and a complementary dual tree leave two discs to sew into a sphere-like surface.
- **Surface classification map.** A concept diagram separates closed orientable surfaces from closed nonorientable surfaces and records the Euler characteristic formulas `2 - 2g` and `2 - n`.
- **Interactive surgery lab.** Plotly displays how handles and crosscaps move a surface through the classification ledger. Interaction helps because the reader can compare genus, crosscap count, orientability, and Euler characteristic without a static printed table.
- **Invariant tests.** Small graph models illustrate how connected components and cut behavior can separate spaces that look similar in a casual drawing.

The notebook uses NetworkX for combinatorial topology, Matplotlib for durable proof and classification diagrams, Plotly for the surgery ledger, and course-local artifact helpers for saved outputs.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_bt_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Basic-Topology course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

UNIT = "chapter-01"
SOURCE_SPAN = {"printed": "1-26", "pdf": "12-37"}
print(f"Course root: {BOOK_ROOT}")
SOURCE_SPAN


In [ ]:

import json
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib, save_plotly_html

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def euler_characteristic(vertices, edges, faces):
    return int(vertices - edges + faces)


def orientable_chi(genus):
    return 2 - 2 * genus


def nonorientable_chi(crosscaps):
    return 2 - crosscaps


def book_rel(path: Path) -> str:
    return Path(path).resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

artifact_paths = []



## Euler Characteristic Is A Topological Ledger

Euler's formula is not merely arithmetic for a favorite drawing. It is a clue that the surface of a sphere-like polyhedron has an invariant count. The ledger `V - E + F` stays at 2 for cell decompositions of the sphere, but it changes for a torus and it behaves additively across disconnected pieces. That is why the hypotheses around Euler's theorem matter.

The first artifact compares several small cell structures. The sphere-like entries have Euler characteristic 2 even when the number of faces changes. The torus entry has Euler characteristic 0. A disconnected pair of sphere-like components has characteristic 4, so connectedness is not a cosmetic condition. The table is deliberately computational: later homology will explain why this same alternating count reappears as an alternating sum of Betti numbers.


In [ ]:

ledger_rows = [
    {"space": "tetrahedron surface", "V": 4, "E": 6, "F": 4, "type": "sphere-like"},
    {"space": "cube surface", "V": 8, "E": 12, "F": 6, "type": "sphere-like"},
    {"space": "icosahedron surface", "V": 12, "E": 30, "F": 20, "type": "sphere-like"},
    {"space": "one-vertex torus cell structure", "V": 1, "E": 2, "F": 1, "type": "torus-like"},
    {"space": "two disjoint tetrahedron surfaces", "V": 8, "E": 12, "F": 8, "type": "disconnected"},
]
for row in ledger_rows:
    row["chi"] = euler_characteristic(row["V"], row["E"], row["F"])
ledger = pd.DataFrame(ledger_rows)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
colors = {"sphere-like": "#2A9D8F", "torus-like": "#E76F51", "disconnected": "#6D597A"}
axes[0].barh(ledger["space"], ledger["chi"], color=[colors[t] for t in ledger["type"]])
axes[0].axvline(2, color="black", linestyle="--", linewidth=1, label="sphere value")
axes[0].set_title("Euler characteristic by cell structure")
axes[0].set_xlabel("V - E + F")
axes[0].legend()

axes[1].axis("off")
table = axes[1].table(
    cellText=ledger[["V", "E", "F", "chi", "type"]].values,
    rowLabels=ledger["space"],
    colLabels=["V", "E", "F", "chi", "type"],
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.35)
axes[1].set_title("The same invariant as an auditable ledger")
fig.suptitle("Chapter 1: Euler characteristic distinguishes surface types", y=1.03, fontsize=14)
fig.tight_layout()
euler_path = save_matplotlib(fig, UNIT, "figures", "euler-characteristic-ledger.png")
plt.close(fig)
artifact_paths.append(euler_path)
display_artifact(euler_path, width=920)

euler_checks = {
    "sphere_like_chi_all_two": bool(all(row["chi"] == 2 for row in ledger_rows if row["type"] == "sphere-like")),
    "torus_chi_zero": bool(ledger.loc[ledger["space"] == "one-vertex torus cell structure", "chi"].iloc[0] == 0),
    "disconnected_pair_chi_four": bool(ledger.loc[ledger["space"] == "two disjoint tetrahedron surfaces", "chi"].iloc[0] == 4),
}
euler_checks



## Proof State: A Tree And A Dual Tree Leave Two Discs

One proof of Euler's theorem is more informative than the formula alone. Choose a spanning tree in the edge graph of a sphere-like polyhedron. The remaining dual edges form a tree in the dual graph. Thickening the two trees gives two discs with a shared boundary; sewing two discs along their boundary is the sphere picture.

The diagram below is not a copied proof figure. It is a small proof-state model. The important inspection target is the dependency: connectedness supplies a spanning tree, the no-nonseparating-loop condition keeps the complementary dual graph tree-like, and the two-disc decomposition explains why a sphere-like surface has Euler characteristic 2.


In [ ]:

proof = nx.DiGraph()
proof_edges = [
    ("connected edge graph", "choose primal spanning tree T"),
    ("no nonseparating loop", "complementary dual edges form tree T*"),
    ("choose primal spanning tree T", "thicken T to a disc"),
    ("complementary dual edges form tree T*", "thicken T* to a second disc"),
    ("thicken T to a disc", "two discs share one boundary"),
    ("thicken T* to a second disc", "two discs share one boundary"),
    ("two discs share one boundary", "sphere-like surface"),
    ("sphere-like surface", "Euler characteristic equals 2"),
]
proof.add_edges_from(proof_edges)
for node in proof.nodes:
    if "connected" in node or "loop" in node:
        proof.nodes[node]["layer"] = 0
    elif "tree" in node:
        proof.nodes[node]["layer"] = 1
    elif "disc" in node:
        proof.nodes[node]["layer"] = 2
    else:
        proof.nodes[node]["layer"] = 3
pos = nx.multipartite_layout(proof, subset_key="layer")
fig, ax = plt.subplots(figsize=(12, 6))
nx.draw_networkx_edges(proof, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=16, alpha=0.45)
nx.draw_networkx_nodes(proof, pos, ax=ax, node_color="#8ecae6", edgecolors="#23343b", node_size=2200)
nx.draw_networkx_labels(proof, pos, ax=ax, font_size=8)
ax.set_title("Proof-state scaffold for Euler's theorem: hypotheses become a two-disc decomposition")
ax.set_axis_off()
fig.tight_layout()
proof_path = save_matplotlib(fig, UNIT, "figures", "euler-proof-tree-dual-tree-scaffold.png")
plt.close(fig)
artifact_paths.append(proof_path)
display_artifact(proof_path, width=920)

proof_checks = {
    "proof_graph_is_acyclic": bool(nx.is_directed_acyclic_graph(proof)),
    "proof_graph_has_euler_sink": "Euler characteristic equals 2" in proof.nodes,
}
proof_checks



## Surfaces: Local Plane, Global Difference

A surface is locally plane-like. That local condition does not decide the global topology. A sphere, torus, double torus, projective plane, Klein bottle, and higher nonorientable surfaces can all be built from local patches while having different global invariants.

The classification theorem preview says that closed connected surfaces fall into two families: orientable surfaces obtained by adding handles to a sphere, and nonorientable surfaces obtained by adding crosscaps. The Euler characteristic formulas summarize the ledger. Handles lower `chi` by 2; crosscaps lower `chi` by 1. The diagram is a route map, not the proof of classification.


In [ ]:

classification_rows = []
for g in range(0, 5):
    classification_rows.append({"family": "orientable", "parameter": g, "description": f"sphere with {g} handles", "chi": orientable_chi(g), "orientable": True})
for n in range(1, 7):
    classification_rows.append({"family": "nonorientable", "parameter": n, "description": f"sphere with {n} crosscaps", "chi": nonorientable_chi(n), "orientable": False})
class_df = pd.DataFrame(classification_rows)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
for family, group in class_df.groupby("family"):
    axes[0].plot(group["parameter"], group["chi"], marker="o", label=family)
axes[0].axhline(0, color="0.4", linewidth=1)
axes[0].set_title("Classification ledger")
axes[0].set_xlabel("handles g or crosscaps n")
axes[0].set_ylabel("Euler characteristic")
axes[0].legend()

route = nx.DiGraph()
route_edges = [
    ("closed connected surface", "orientable?"),
    ("orientable?", "sphere with g handles"),
    ("orientable?", "sphere with n crosscaps"),
    ("sphere with g handles", "chi = 2 - 2g"),
    ("sphere with n crosscaps", "chi = 2 - n"),
    ("sphere with g handles", "two-sided local orientation survives loops"),
    ("sphere with n crosscaps", "some loop reverses local orientation"),
]
route.add_edges_from(route_edges)
pos2 = nx.spring_layout(route, seed=14, k=1.0)
colors2 = ["#2A9D8F" if "handles" in node or "orientation survives" in node else "#E76F51" if "crosscaps" in node or "reverses" in node else "#8ecae6" for node in route.nodes]
nx.draw_networkx_edges(route, pos2, ax=axes[1], arrows=True, arrowstyle="-|>", arrowsize=14, alpha=0.45)
nx.draw_networkx_nodes(route, pos2, ax=axes[1], node_color=colors2, edgecolors="#23343b", node_size=1800)
nx.draw_networkx_labels(route, pos2, ax=axes[1], font_size=8)
axes[1].set_title("Classification theorem preview")
axes[1].set_axis_off()
fig.suptitle("Closed surfaces are organized by orientability and one integer", y=1.03, fontsize=14)
fig.tight_layout()
classification_path = save_matplotlib(fig, UNIT, "figures", "surface-classification-ledger.png")
plt.close(fig)
artifact_paths.append(classification_path)
display_artifact(classification_path, width=920)

classification_checks = {
    "torus_genus_one_chi_zero": bool(orientable_chi(1) == 0),
    "sphere_genus_zero_chi_two": bool(orientable_chi(0) == 2),
    "projective_plane_crosscap_one_chi_one": bool(nonorientable_chi(1) == 1),
    "klein_bottle_crosscap_two_chi_zero": bool(nonorientable_chi(2) == 0),
}
classification_checks



## Applied Lab: Handles, Crosscaps, And Invariant-Ledger Navigation

The interactive lab is a compact version of the classification preview. Choose a family, move the parameter, and watch the Euler characteristic and orientability flag change. This is not a surface renderer. It is an invariant dashboard: the same style of dashboard will reappear when homology turns the Euler characteristic into an alternating rank count.

Use the lab to compare surfaces that share `chi` but differ in orientability. For example, the torus and the Klein bottle both have Euler characteristic 0, yet they are not homeomorphic because orientability separates them. This shows the limits of a single invariant and motivates using a suite of invariants.


In [ ]:

parameters = np.arange(0, 7)
orientable_chis = np.array([orientable_chi(g) for g in parameters])
nonorientable_ns = np.arange(1, 8)
nonorientable_chis = np.array([nonorientable_chi(n) for n in nonorientable_ns])
fig = go.Figure()
fig.add_trace(go.Scatter(x=parameters, y=orientable_chis, mode="lines+markers", name="orientable: sphere + g handles", hovertemplate="g=%{x}<br>chi=%{y}<br>orientable=yes<extra></extra>"))
fig.add_trace(go.Scatter(x=nonorientable_ns, y=nonorientable_chis, mode="lines+markers", name="nonorientable: sphere + n crosscaps", hovertemplate="n=%{x}<br>chi=%{y}<br>orientable=no<extra></extra>"))
fig.add_hline(y=0, line_dash="dash", line_color="#888888")
fig.update_layout(
    title="Chapter 1 interactive surgery ledger: handles and crosscaps change invariants",
    xaxis_title="surface parameter",
    yaxis_title="Euler characteristic",
    height=520,
)
surgery_path = save_plotly_html(fig, UNIT, "html", "surface-surgery-invariant-ledger.html", include_plotlyjs=True)
artifact_paths.append(surgery_path)
display_artifact(surgery_path, width="100%", height=540)

surgery_checks = {
    "handle_decreases_chi_by_two": bool(np.all(np.diff(orientable_chis) == -2)),
    "crosscap_decreases_chi_by_one": bool(np.all(np.diff(nonorientable_chis) == -1)),
    "torus_and_klein_share_chi_not_orientability": bool(orientable_chi(1) == nonorientable_chi(2) == 0),
}
surgery_checks



## Invariants Separate Spaces When Pictures Are Ambiguous

A homeomorphism is a continuous bijection with continuous inverse. Drawing one explicitly can be hard; proving no homeomorphism exists is usually harder. Topological invariants are the practical escape route. If two spaces have different Euler characteristic, connectedness, orientability, or cut behavior, they cannot be homeomorphic.

The miniature graph models below are not substitutes for the full spaces. They are inspection tools for the logic of invariants. Removing a point from a circle-like cycle keeps it connected as a path; removing a separating point from a figure-eight graph disconnects the graph. A single invariant may fail to distinguish two spaces, so the course builds a toolbox rather than a single magic number.


In [ ]:

cycle = nx.cycle_graph(12)
figure_eight = nx.Graph()
figure_eight.add_edges_from([(0,1),(1,2),(2,3),(3,0),(0,4),(4,5),(5,6),(6,0)])

def components_after_removing(G, node):
    H = G.copy()
    H.remove_node(node)
    return nx.number_connected_components(H)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0))
for ax, G, title in [(axes[0], cycle, "cycle model"), (axes[1], figure_eight, "figure-eight model")]:
    pos = nx.circular_layout(G) if G is cycle else nx.spring_layout(G, seed=8)
    nx.draw_networkx_edges(G, pos, ax=ax, width=1.8, alpha=0.6)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=["#E76F51" if n == 0 else "#8ecae6" for n in G.nodes], edgecolors="#23343b")
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
    ax.set_title(title)
    ax.set_axis_off()
fig.suptitle("Invariant habit: removing a point can reveal different connectivity behavior", y=1.02, fontsize=14)
fig.tight_layout()
invariant_path = save_matplotlib(fig, UNIT, "figures", "connectedness-cutpoint-invariant.png")
plt.close(fig)
artifact_paths.append(invariant_path)
display_artifact(invariant_path, width=900)

invariant_checks = {
    "cycle_minus_one_point_connected": bool(components_after_removing(cycle, 0) == 1),
    "figure_eight_center_removal_disconnects": bool(components_after_removing(figure_eight, 0) == 2),
    "connectedness_is_preserved_in_model_examples": bool(nx.is_connected(cycle) and nx.is_connected(figure_eight)),
}
invariant_checks



## Reading The Checks

The checks in this notebook are intentionally elementary. They should be read as a contract for the chapter. Euler characteristic distinguishes sphere-like and torus-like examples, but it does not distinguish every surface. Orientability adds another invariant. Connectedness and cut behavior add more. A topologist's job is often to choose an invariant that survives homeomorphism but changes between the spaces in question.

The source chapter begins with concrete polyhedra and surfaces because the abstraction is easier to trust after the invariant pattern is visible. Later chapters will formalize continuity, compactness, connectedness, quotient spaces, fundamental groups, triangulations, homology, degree, and covering spaces. This first notebook gives the map: topology studies what remains after deformation, and invariants are how we test that remaining structure.


In [ ]:

all_checks = {
    **euler_checks,
    **proof_checks,
    **classification_checks,
    **surgery_checks,
    **invariant_checks,
}
assert all(value for value in all_checks.values() if isinstance(value, bool)), all_checks
for artifact in artifact_paths:
    assert_artifact(artifact, min_bytes=512)
    artifact.resolve().relative_to((BOOK_ROOT / "artifacts" / UNIT).resolve())
checks_path = save_json({"source_span": SOURCE_SPAN, "checks": all_checks, "artifacts": [book_rel(p) for p in artifact_paths]}, UNIT, "checks", "chapter-01-invariants.json")
artifact_paths.append(checks_path)
display_artifact(checks_path)
all_checks



## Takeaways

Euler characteristic is the chapter's first computational invariant. It turns a surface decomposition into a number, and it already shows why hypotheses such as connectedness and sphere-like loop behavior matter. Homeomorphism is the equivalence relation behind this calculation: surfaces that can be continuously deformed into one another should have the same topological invariants.

Closed surfaces are previewed by two families: orientable surfaces built by adding handles and nonorientable surfaces built by adding crosscaps. The formulas `chi = 2 - 2g` and `chi = 2 - n` are classification signposts, not the whole proof. Orientability, connectedness, and cut behavior remind us that one invariant rarely tells the whole story. The practical habit is to pair each visual claim with an invariant that can be checked.


In [ ]:

final_sanity = {
    "source_span": SOURCE_SPAN,
    "artifacts": [book_rel(assert_artifact(path, min_bytes=512)) for path in artifact_paths],
    "core_checks": {
        "sphere_like_chi_two": euler_checks["sphere_like_chi_all_two"],
        "torus_chi_zero": euler_checks["torus_chi_zero"],
        "classification_formulas": all(classification_checks.values()),
        "surgery_deltas": surgery_checks["handle_decreases_chi_by_two"] and surgery_checks["crosscap_decreases_chi_by_one"],
        "connectedness_cutpoint_examples": invariant_checks["cycle_minus_one_point_connected"] and invariant_checks["figure_eight_center_removal_disconnects"],
    },
    "pdf_used_for": "source orientation only; no copied prose, exercises, screenshots, page crops, or figures",
    "standalone_contract": "Euler ledger, proof-state scaffold, surface classification map, surgery lab, invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, UNIT, "checks", "final-sanity.json")
assert_artifact(final_path, min_bytes=512)
final_sanity
